# AI School · Epoch 2 · Checkpoint 2.1
## Linear Regression on Bike-Sharing Demand

---

**Total Points:** 100 points (main) + 10 bonus points  
**Minimum Required:** 85 / 100

**Dataset:** Bike Sharing Demand — loaded directly from Hugging Face 🤗  
**Goal:** Build, evaluate, and interpret Linear Regression and its regularised variants (Ridge & Lasso) on a real-world regression task.

**Submission:** Push your `.ipynb` to GitHub and submit the repo link in the form: https://forms.gle/r3JLHDKfxR5XAcdr5.

> *Take your time on the reflection questions — they matter just as much as the code.*

---

## Part 0 · Setup

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import learning_curve, cross_val_score

sns.set_theme(style="whitegrid")
%matplotlib inline

---

## Part 1 · Conceptual Warm-Up  *(15 points)*

Before touching any code, take a moment to reflect.  
Write your answers in the Markdown cells provided — a few clear sentences per question is enough.

### Q1 · What is Hugging Face? *(5 points)*

In your own words, answer the following:

1. What is Hugging Face and what role does it play in the modern ML ecosystem?
2. Beyond hosting datasets, name **three** functionalities it offers.
3. Why is using a centralised hub like HF better than emailing CSV files around a team?

✏️ *Write your answer below:*

## Answer : 

1. Hugging Face is a platform that offers and shares open-source AI models. It helps developers accelerate the development of machine learning systems by providing access to, modification of, and improvement of the code. It is also a collaborative repository where developers and researchers can use datasets

2. Hugging Face allows to : share AI models, collaborate on projects, and execute scripts

3. using a centralised hub like HF is better because everyone accesses the data in the same place, and several people can modify and download files without sending them

### Q2 · Log Transformation vs. Scalers *(5 points)*

In the preprocessing notebook, the target `cnt` was replaced with `log_cnt = log1p(cnt)`.

1. What is a log transformation, and what property of a distribution does it correct?
2. When should you apply it (give a concrete rule of thumb)?
3. How is it **fundamentally different** from StandardScaler or MinMaxScaler?  
   *(Hint: think about what each one changes — the shape of the distribution or just its location/scale.)*

✏️ *Write your answer below:*

## Answer : 

1. The log transformation uses this formula : y′=log(y+1) (y is the target variable), in order to reduce asymmetry and stabilize variance, it also allows the model to better understand the underlying trends in the data, making predictions more accurate

2. We apply the log transformation for the target when we have a non-linear relationship which indicates that our distribution is not normal, and also when the skew is far from 0

3. Log transformation and MinMaxScaler are both data transformation methods. However, log transformation changes the shape of the data distribution and helps handle non-linear relationships, while MinMaxScaler only rescales the features so that they fit within a specific range, usually [0,1], and does not change the shape 

### Q3 · Personal Reflection *(5 points)*

Look back at the two EDA & preprocessing notebooks.

1. Which preprocessing step was the most **novel** to you, and why?
2. Were there any decisions you would have made differently? Justify your answer.

✏️ *Write your answer below:*

## Answer : 

---

## Part 2 · Loading Data from Hugging Face  *(10 points)*

The dataset has already been cleaned, preprocessed, and uploaded to Hugging Face for you.  
Your first task is to load it — this is a skill you will use in every future project.

### Exo 1 · Install & import the HF library *(2 points)*

Run the cell below to install `datasets` (already done in Colab — just make sure it works).

In [6]:
!pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/914.9 kB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 4.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 3.4 MB/s eta 0:00:01
   -------------- ------------------------- 0.8/2.2 MB 3.4 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.2 MB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 2.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
!pip install datasets -q   
from datasets import load_dataset


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Exo 2 · Load the dataset *(4 points)*

Use `load_dataset` to load `"b-fatma/bike-sharing-federated"`.  
The dataset has two splits: `"train"` and `"test"`.

Convert both splits to pandas DataFrames and store them in:  
- `train_df`, `test_df`

Then **print all column names** and the shape of each DataFrame.  
Identify which column is the target (it will be the one that is not a feature).

> 💡 **Hint:** The feature columns you expect are the ones produced by the preprocessing notebook  
> (`yr`, `workingday`, `atemp`, `hum`, `windspeed`, `hr_sin`, `hr_cos`, and the one-hot encoded  
> `season_*` and `weathersit_*` columns). Any extra column is the target.

In [8]:
# load the dataset from HF
dataset = load_dataset("b-fatma/bike-sharing-federated")

C:\Users\Sarah\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sarah\.cache\huggingface\hub\datasets--b-fatma--bike-sharing-federated. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 3476/3476 [00:00<00:00, 41838.92 examples/s]


In [12]:
# convert to DataFrames and print shapes
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

print(" Train Column :" , train_df.columns)
print("\n Train Shape :" , train_df.shape)

print("\n Test Column :" , test_df.columns)
print("\n Test Shape :" , test_df.shape)

 Train Column : Index(['yr', 'workingday', 'atemp', 'hum', 'windspeed', 'hr_sin', 'hr_cos',
       'season_2', 'season_3', 'season_4', 'weathersit_2', 'weathersit_3',
       'cnt_log'],
      dtype='str')

 Train Shape : (13900, 13)

 Test Column : Index(['yr', 'workingday', 'atemp', 'hum', 'windspeed', 'hr_sin', 'hr_cos',
       'season_2', 'season_3', 'season_4', 'weathersit_2', 'weathersit_3',
       'cnt_log'],
      dtype='str')

 Test Shape : (3476, 13)


### Exo 3 · Separate features and target *(4 points)*

After inspecting the columns in Exo 2, identify the **target column** (the log-transformed count).  
Store its name in a variable called `TARGET`.

Split each DataFrame into `X` (features) and `y` (target):
- `X_train`, `y_train`
- `X_test`, `y_test`

Print `TARGET` and the shape of `X_train`.

In [13]:
# Inspect all columns to identify the target
print(train_df.columns.tolist())

['yr', 'workingday', 'atemp', 'hum', 'windspeed', 'hr_sin', 'hr_cos', 'season_2', 'season_3', 'season_4', 'weathersit_2', 'weathersit_3', 'cnt_log']


In [14]:
# Set TARGET to the column name you identified above, then split
TARGET = 'cnt_log'   # replace ??? with the actual column name

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET].values

X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET].values

print(f'Target column: {TARGET}')
print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')

Target column: cnt_log
X_train: (13900, 12)  |  y_train: (13900,)


---

## Part 3 · Baseline Linear Regression  *(30 points)*

### Exo 4 · Fit a Linear Regression model *(5 points)*

Using **scikit-learn**:
1. Instantiate a `LinearRegression` model.
2. Fit it on `X_train`, `y_train`.
3. Generate predictions on both the **train** and **test** sets.

Store predictions as `y_train_pred` and `y_test_pred`.

In [ ]:
# fit LinearRegression

In [ ]:
# predict on train and test

### Exo 5 · Compute & interpret metrics *(8 points)*

Compute the following metrics for **both** train and test sets and display them in a clean DataFrame:

| Set   | MSE | RMSE | MAE | R² |
|-------|-----|------|-----|----|

Then answer (in Markdown below the table):
- What does each metric tell you?
- Is the model underfitting, overfitting, or well-fitted? How can you tell from these numbers?

In [ ]:
# compute metrics

✏️ *Interpret the metrics here:*

### Exo 6 · Residual plot *(7 points)*

A **residual** is the difference between the true value and the predicted value: `residual = y_true - y_pred`.

1. Compute residuals for the **test** set.
2. Create a scatter plot: predicted values (x-axis) vs residuals (y-axis).
3. Add a horizontal dashed red line at `y = 0`.
4. Label your axes and add a title.

Then answer in Markdown:
- What pattern do you see (if any)?
- What would a *perfect* residual plot look like?
- What does a funnel shape indicate?

In [ ]:
# compute residuals

In [ ]:
# residual plot

✏️ *Interpret the residual plot here:*

### Exo 7 · Actual vs Predicted plot *(5 points)*

1. Create a scatter plot: `y_test` (x-axis) vs `y_test_pred` (y-axis).
2. Add a **diagonal line** (y = x) in red — this represents a perfect model.
3. Label axes and add a title.

Then answer:
- How close are the dots to the diagonal line?
- Where does the model struggle the most?

In [ ]:
# actual vs predicted plot

✏️ *Interpret here:*

### Exo 8 · Learning curve *(5 points)*

A learning curve shows how training and validation scores evolve as the training set grows.

Use `sklearn.model_selection.learning_curve` with `cv=5` and `scoring='neg_mean_squared_error'`.

Plot:
- Training score (blue)
- Validation score (orange)
- Add a shaded region for ±1 std

Then answer:
- Do the curves converge? What does that mean?
- Does more data help this model?

In [ ]:
# compute learning curves

In [ ]:
# plot learning curves

✏️ *Interpret here:*

---

## Part 4 · Feature Importance via Coefficients  *(15 points)*

### Exo 9 · Extract and visualise coefficients *(10 points)*

In a linear model, the **coefficients** tell us how much each feature contributes to the prediction (after scaling).

1. Extract the coefficients from your fitted model using `.coef_`.
2. Create a **horizontal bar plot** — features on the y-axis, coefficient values on the x-axis.
3. Sort features by the absolute value of their coefficient (largest at the top).
4. Use **green** for positive coefficients and **red** for negative ones.

Then answer:
- Which features push the prediction **up**? Which pull it **down**?
- Does this make intuitive sense given what you know about bike rentals?

In [ ]:
# extract coefficients

In [ ]:
# barplot of coefficients

✏️ *Interpret here:*

### Exo 10 · Coefficient magnitude vs feature importance *(5 points)*

Answer the following in Markdown:

1. Why do we need features to be **scaled** before comparing coefficients?
2. Can we always trust coefficient magnitude as a proxy for feature importance? What could go wrong with correlated features?
3. Name one alternative method for computing feature importance that is model-agnostic.

✏️ *Write your answer below:*

---

## Part 5 · Regularisation — Ridge & Lasso  *(20 points)*

When models overfit, regularisation adds a **penalty** on large coefficients to constrain the model.

- **Ridge (L2):** penalises the sum of squared coefficients → shrinks all coefficients smoothly  
- **Lasso (L1):** penalises the sum of absolute coefficients → can shrink some coefficients **all the way to zero** (automatic feature selection)

Both have a hyperparameter `alpha` (sometimes called **λ**) that controls the strength of the penalty.

### Exo 11 · Fit Ridge and Lasso with default alpha *(6 points)*

1. Fit a `Ridge(alpha=1.0)` and a `Lasso(alpha=1.0)` on the training data.
2. Compute the same four metrics (MSE, RMSE, MAE, R²) for all three models (Linear, Ridge, Lasso) on the **test** set.
3. Display results in a single comparison table.

In [ ]:
# fit Ridge and Lasso

In [ ]:
# comparison metrics table

### Exo 12 · Alpha (λ) tuning for Ridge *(7 points)*

Let's explore how `alpha` affects Ridge performance.

1. Define a list of alphas: `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`
2. For each alpha, fit Ridge and record **train R²** and **test R²**.
3. Plot both curves on the same graph (alpha on the x-axis on a **log scale**).

Then answer:
- What happens to the model as alpha → 0?
- What happens as alpha → ∞?
- Which alpha gives the best balance?

> 💡 **Hint:** Use `plt.xscale('log')` and mark the best alpha with a vertical dashed line.

In [ ]:
# alpha sweep for Ridge

In [ ]:
# plot R² vs alpha

✏️ *Interpret here:*

### Exo 13 · Lasso and sparsity *(7 points)*

Lasso is special because it can drive coefficients to exactly zero.

1. Fit Lasso models for the same list of alphas as above.
2. For each alpha, record the **number of non-zero coefficients**.
3. Plot: alpha (log scale, x-axis) vs number of non-zero coefficients (y-axis).

Then answer:
- At which alpha do most coefficients disappear?
- What does this tell us about Lasso as a feature selection method?
- Would you prefer Lasso or Ridge for this dataset, and why?

In [ ]:
# Lasso sparsity sweep

In [ ]:
# plot sparsity vs alpha

✏️ *Interpret here:*

---

## Part 6 · Final Model Comparison  *(10 points)*

### Exo 14 · Best model selection *(5 points)*

Based on your experiments:

1. Choose the **best alpha** for Ridge and Lasso (justify your choice).
2. Refit both with the best alphas.
3. Build a final comparison table: Linear Regression vs Ridge (best α) vs Lasso (best α)  
   Include: Test MSE, Test RMSE, Test MAE, Test R², and number of non-zero coefficients.

In [ ]:
# refit best Ridge and Lasso

In [ ]:
# final comparison table

### Exo 15 · Coefficient comparison plot *(5 points)*

Create a **side-by-side bar chart** showing the coefficients for all three models (LR, Ridge, Lasso) for each feature.

- Use different colours for each model
- Sort by the absolute value of Linear Regression coefficients

Then answer: What is the most visible effect of regularisation on the coefficients?

In [ ]:
# coefficient comparison barplot

✏️ *Interpret here:*

---

## Bonus · *(10 points)*

### Bonus 1 · Cross-Validated Alpha Tuning *(5 points)*

Instead of a manual loop, use `sklearn.linear_model.RidgeCV` and `LassoCV`, which perform built-in cross-validated alpha selection.

1. Fit `RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)` on the training set.
2. Print the best alpha chosen.
3. Compute test R² for this model and compare to your manual best Ridge from Exo 12.
4. Repeat for `LassoCV`.

In [ ]:
# RidgeCV and LassoCV

### Bonus 2 · Predict on Original Scale *(5 points)*

Remember that we log-transformed the target. Predictions are currently in log-space.

1. Convert predictions back to the **original count scale** using `np.expm1()`.
2. Compute RMSE in the original scale for your best model.
3. Plot Actual vs Predicted in the original scale.

> 💡 Why is it important to report metrics in the original scale when communicating results to a non-technical audience?

In [ ]:
# inverse transform and original-scale evaluation

✏️ *Interpret here:*

---

---

## 🔧 Optional Part · Linear Regression from Scratch

> This section is **optional** — it will not affect your score. But if you want to truly understand what sklearn is doing under the hood, this is the place to find out.

We are going to build a complete `LinearRegressionScratch` class using nothing but **NumPy**. Every step is broken into a small guided exercise so you always know what to implement next.

By the end you will:
- Understand the **Normal Equation** (the closed-form solution to linear regression)
- Implement **Gradient Descent** from first principles
- Plot the **loss curve** to watch the model learn
- Compare your results to sklearn's — they should be virtually identical ✅


---

### 🧮 Step 1 · The Math — read carefully before coding

Linear regression fits a line (or hyperplane) to data by finding weights **w** and bias **b** such that:

```
ŷ = Xw + b
```

We absorb the bias into **w** by adding a column of ones to X (called the **bias trick**):

```
X̃ = [1 | X]   →   ŷ = X̃ w̃
```

**Two ways to find w̃:**

#### Option A — Normal Equation (closed-form)
Minimise the Mean Squared Error directly:

```
MSE = (1/n) ‖ ŷ - y ‖²
```

Taking the derivative and setting it to zero gives:

```
w̃* = (X̃ᵀ X̃)⁻¹ X̃ᵀ y
```

One matrix operation — exact, no iterations. Works well when the number of features is small (< ~10k).

#### Option B — Gradient Descent (iterative)
Update the weights step-by-step in the direction that reduces the loss:

```
gradient = (2/n) X̃ᵀ (ŷ - y)
w̃ ← w̃ - lr × gradient
```

Repeat for `n_iters` iterations. Essential when the dataset is too large to invert the matrix.


---

### 🏗️ Step 2 · Build the class skeleton *(read only — do not edit)*

Below is the skeleton of our class. The `__init__`, `_add_bias`, and `predict` methods are already written for you. You only need to implement `fit_normal_equation` and `fit_gradient_descent`.


In [ ]:
class LinearRegressionScratch:
    """
    Linear Regression implemented from scratch with NumPy.
    Supports two solvers: 'normal_equation' and 'gradient_descent'.
    """

    def __init__(self, solver='normal_equation', lr=0.01, n_iters=1000):
        """
        Parameters
        ----------
        solver   : 'normal_equation' or 'gradient_descent'
        lr       : learning rate (only used by gradient descent)
        n_iters  : number of iterations (only used by gradient descent)
        """
        self.solver   = solver
        self.lr       = lr
        self.n_iters  = n_iters
        self.weights  = None   # shape: (n_features + 1,)  — includes bias
        self.losses   = []     # loss history (gradient descent only)

    # ── internal helper ────────────────────────────────────────
    def _add_bias(self, X):
        """Prepend a column of ones to X so the bias is part of the weight vector."""
        ones = np.ones((X.shape[0], 1))
        return np.hstack([ones, X])

    # ── predict ────────────────────────────────────────────────
    def predict(self, X):
        """Return predictions for X (does not include the bias column yet)."""
        X_b = self._add_bias(X)
        return X_b @ self.weights

    # ── fit ────────────────────────────────────────────────────
    def fit(self, X, y):
        if self.solver == 'normal_equation':
            self.fit_normal_equation(X, y)
        elif self.solver == 'gradient_descent':
            self.fit_gradient_descent(X, y)
        else:
            raise ValueError(f"Unknown solver: {self.solver}")
        return self

    # ── solvers ── you implement these ─────────────────────────
    def fit_normal_equation(self, X, y):
        raise NotImplementedError("Implement me!")

    def fit_gradient_descent(self, X, y):
        raise NotImplementedError("Implement me!")

---

### ✏️ Step 3 · Implement the Normal Equation *(guided)*

Fill in `fit_normal_equation` below.

The formula is:   **w̃ = (X̃ᵀ X̃)⁻¹ X̃ᵀ y**

Useful NumPy functions:
- `np.linalg.inv(A)` — matrix inverse  
- `A.T` — transpose  
- `A @ B` — matrix multiply

> 💡 **Hint:** The steps are exactly: add bias → compute Xᵀ X → invert it → multiply by Xᵀ y.


In [ ]:
class LinearRegressionScratch(LinearRegressionScratch):

    def fit_normal_equation(self, X, y):
        X_b = self._add_bias(X)          # shape: (n, p+1)

        # TODO: compute w = (X_b^T X_b)^{-1} X_b^T y
        # Store the result in self.weights
        pass

---

### ✏️ Step 4 · Implement Gradient Descent *(guided)*

Fill in `fit_gradient_descent` below.

Algorithm (repeat `n_iters` times):
```
1. Compute predictions:  ŷ = X̃ @ w̃
2. Compute error:        e = ŷ - y
3. Compute gradient:     grad = (2/n) * X̃ᵀ @ e
4. Update weights:       w̃ = w̃ - lr * grad
5. Record loss:          loss = mean(e²)   → append to self.losses
```

> 💡 **Hint:** Initialise `self.weights` to a vector of zeros with shape `(n_features + 1,)` before the loop.


In [ ]:
class LinearRegressionScratch(LinearRegressionScratch):

    def fit_gradient_descent(self, X, y):
        X_b = self._add_bias(X)          # shape: (n, p+1)
        n, p = X_b.shape
        self.weights = np.zeros(p)       # initialise weights to zero
        self.losses  = []

        for _ in range(self.n_iters):
            # TODO: step 1 — predictions
            y_pred = None

            # TODO: step 2 — error
            error = None

            # TODO: step 3 — gradient
            grad = None

            # TODO: step 4 — weight update
            # self.weights = ...

            # TODO: step 5 — record MSE loss
            loss = None
            self.losses.append(loss)

---

### 📉 Step 5 · Plot the Loss Curve

Once gradient descent is implemented, run it and plot the loss over iterations.

1. Fit a `LinearRegressionScratch(solver='gradient_descent', lr=0.01, n_iters=1000)` on the training data.
2. Plot `model_gd.losses` vs iteration index.
3. Label axes and add a title.

Then answer in Markdown:
- Does the loss go down smoothly? What would a jagged loss curve indicate?
- What happens if you set `lr=10`? Try it and describe what you observe.


In [ ]:
# fit gradient descent model

In [ ]:
# plot loss curve

✏️ *Interpret here:*

---

### 🔬 Step 6 · Sanity-check both solvers

Run both solvers and verify they produce the same weights.

1. Fit `LinearRegressionScratch(solver='normal_equation')` → `model_ne`
2. Fit `LinearRegressionScratch(solver='gradient_descent', lr=0.01, n_iters=2000)` → `model_gd`
3. Print the first 5 weights from each model side-by-side.
4. Check that they are close with `np.allclose(model_ne.weights, model_gd.weights, atol=1e-2)`.

> 💡 They won't be *exactly* equal — gradient descent is an approximation — but they should be very close.


In [ ]:
# fit both solvers

In [ ]:
# compare weights

---

### 📊 Step 7 · Final Comparison — Scratch vs sklearn

Now let's see how your implementation stacks up against sklearn.

1. Compute test MSE, RMSE, MAE, and R² for:
   - `model_ne` (your Normal Equation)
   - `model_gd` (your Gradient Descent)
   - `lr` (sklearn's LinearRegression from Part 3)

2. Display all three in a single comparison table.

3. Answer in Markdown:
   - Are the metrics essentially the same? Why (or why not)?
   - What advantages does sklearn have over your implementation?
   - In what situations would gradient descent be preferred over the normal equation?


In [ ]:
# compute metrics for all three models

In [ ]:
# comparison table

✏️ *Interpret here:*

---

### 🎁 Bonus Challenge (no points — just glory)

Extend your `LinearRegressionScratch` class to support **Ridge regularisation**:

The gradient descent update becomes:

```
grad = (2/n) X̃ᵀ (ŷ - y) + 2λ w̃
```

(but **do not** regularise the bias term — index 0 of `self.weights`)

Test it with `λ = 1.0` and compare to sklearn's `Ridge(alpha=1.0)`.


In [ ]:
# Ridge from scratch — extend your class here

## Acknowledgments

- Notebook authored by: BOULGHERAIF Fatma Zohra (GitHub: b-fatma), Open Minds Club — AI Leadership  
- Dataset: [UCI Bike Sharing Dataset](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset)  
- Preprocessed and hosted on [Hugging Face](https://huggingface.co/datasets/b-fatma/bike-sharing-federated)